# S22 · Why a bowl is easy

We draw the two pictures that decide whether a "find the best answer" problem is
easy or dangerous: a shape with no dents (a **convex set**), a bowl-shaped cost (a
**convex function**), and why a bowl is so much friendlier than a wobbly landscape
full of traps.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. This one is almost all
  drawing. Press the play button on each cell, top to bottom, and read the
  plain-English note above each one.
- New to the "bowl vs wobbly" idea? Open the primer
  `primers/convex_sets_and_functions.md` for a ten-minute, picture-first version.
- Already confident with the maths? The pictures are still worth a look, then jump
  to the next two notebooks where we actually solve real problems.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook only uses NumPy and Matplotlib, which Colab already ships.
# So there is nothing to install here.
print("Setup complete - nothing to install.")

## Step 1 — import the tools we need

Just NumPy for a bit of maths and Matplotlib for the pictures. Everything in this
notebook is drawing and checking, so there is no heavy computation.

In [ ]:
import numpy as np                 # arrays and maths on lists of numbers
import matplotlib.pyplot as plt    # drawing charts

# Set a seed so any random numbers are the same every time we run this.
np.random.seed(0)

print("Tools ready.")

## Step 2 — a shape with no dents, and one with a dent

Think of a shape as the set of answers you are allowed to pick. The shape is
**convex** if it has no dents. The test is simple: pick any two points inside it
and draw the straight line between them. If that line always stays inside, the
shape is convex.

We draw a filled disc (no dents) next to a crescent moon (one big dent), and drop a
line between two points on each so you can see the difference.

In [ ]:
# Make a figure with two pictures side by side.
figure, axes = plt.subplots(1, 2, figsize=(11, 5))

# --- Left picture: a disc, which has no dents (convex) ---
left = axes[0]

# A circle drawn from angles going all the way around.
angle = np.linspace(0, 2 * np.pi, 200)
circle_x = np.cos(angle)
circle_y = np.sin(angle)
left.fill(circle_x, circle_y, color="#D6E4F0")   # fill the disc light blue

# Pick two points inside the disc and join them with a line.
point_a = (-0.6, -0.3)
point_b = (0.6, 0.4)
left.plot([point_a[0], point_b[0]], [point_a[1], point_b[1]],
          color="#1E8449", linewidth=3)
left.scatter([point_a[0], point_b[0]], [point_a[1], point_b[1]],
             color="#2E75B6", s=60, zorder=5)
left.set_title("No dents: the line stays inside")
left.set_aspect("equal")
left.axis("off")

# --- Right picture: a crescent, which has a dent (not convex) ---
right = axes[1]

# Draw the outer disc, then "carve a bite" out of it with a white disc.
outer_x = 1.3 * np.cos(angle)
outer_y = 1.0 * np.sin(angle)
right.fill(outer_x, outer_y, color="#F2D7D5")     # the crescent body
bite_x = 0.6 + 0.7 * np.cos(angle)
bite_y = 0.7 * np.sin(angle)
right.fill(bite_x, bite_y, color="white")          # the bite taken out

# Pick two points so the line between them leaves the shape.
point_c = (-0.2, 0.7)
point_d = (-0.2, -0.7)
right.plot([point_c[0], point_d[0]], [point_c[1], point_d[1]],
           color="#C0392B", linewidth=3, linestyle="--")
right.scatter([point_c[0], point_d[0]], [point_c[1], point_d[1]],
              color="#2E75B6", s=60, zorder=5)
right.set_title("A dent: the line leaves it")
right.set_aspect("equal")
right.axis("off")

plt.show()

## Step 3 — what we just saw

On the left, every line between two points stays inside the disc, so the disc is
convex. On the right, the dashed line cuts across the empty bite, so it leaves the
shape. That one bad pair of points is enough to make the whole shape non-convex.

This matters because the set of allowed answers in many real problems turns out to
have no dents, and dent-free shapes are the well-behaved ones to search over.

## Step 4 — a bowl-shaped cost

Now put a cost on top of the shape: high ground is expensive, low ground is cheap,
and we want the lowest point. A cost is **convex** if its graph curves up like a
bowl.

The test again uses a straight line. Pick any two points on the curve and join them
with a straight line, called a *chord*. For a bowl the chord always lies **on or
above** the curve. We plot `f(x) = x²`, the simplest bowl, and draw one such
chord.

In [ ]:
# A grid of x values to draw the curve smoothly.
x_values = np.linspace(-2.5, 2.5, 200)

# The bowl-shaped cost: the simplest one there is.
f_values = x_values ** 2

# Pick two points on the curve to draw a chord between.
left_x = -1.8
right_x = 2.0
left_y = left_x ** 2
right_y = right_x ** 2

plt.figure(figsize=(7, 5))

# The curve itself.
plt.plot(x_values, f_values, color="#2E75B6", linewidth=3,
         label="bowl-shaped f(x) = x^2")

# The chord joining the two chosen points.
plt.plot([left_x, right_x], [left_y, right_y],
         color="#C0392B", linewidth=2.5, label="chord (stays above)")
plt.scatter([left_x, right_x], [left_y, right_y], color="#1E8449", s=60, zorder=5)

plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("A bowl: every chord lies above the curve")
plt.legend()
plt.show()

## Step 5 — check the "chord stays above" claim with numbers

We do not have to trust our eyes. We can take several points between the two ends,
work out the height of the chord and the height of the curve at each, and confirm
the chord is never below the curve. A simple `for` loop does it.

In [ ]:
# Look at 9 evenly spaced points between left_x and right_x.
test_points = np.linspace(left_x, right_x, 9)

print("   x      curve     chord    chord above curve?")
chord_is_always_above = True

for x in test_points:
    # Height of the curve at this x.
    curve_height = x ** 2

    # Height of the straight chord at this x.
    # The chord runs from (left_x, left_y) to (right_x, right_y).
    fraction = (x - left_x) / (right_x - left_x)   # 0 at the left end, 1 at the right
    chord_height = left_y + fraction * (right_y - left_y)

    is_above = chord_height >= curve_height - 1e-9   # allow a tiny rounding wiggle
    if not is_above:
        chord_is_always_above = False

    print(f"{x:6.2f}   {curve_height:7.3f}   {chord_height:7.3f}      {is_above}")

print()
print("Chord stayed above the curve everywhere:", chord_is_always_above)

## Step 6 — why a bowl beats a wobbly landscape

Finding the best answer means finding the lowest point of a cost. We compare a bowl
(one single bottom) with a wobbly landscape (lots of little dips). Imagine dropping
a marble and letting it roll downhill until it stops.

On the bowl it always rolls to the one bottom. On the wobbly one it settles in
whatever little dip it happens to be near, which may not be the true lowest point at
all.

In [ ]:
# A grid of decision values (the thing we are choosing).
x = np.linspace(-3, 3, 400)

# The bowl: one smooth valley.
bowl = 0.5 * x ** 2

# The wobbly landscape: a bowl with wiggles added so it has many local dips.
wobbly = 0.5 * x ** 2 + 1.6 * np.sin(3 * x) + 0.5 * np.cos(7 * x)

figure, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Left: the bowl ---
left = axes[0]
left.plot(x, bowl, color="#2E75B6", linewidth=3)
# The single lowest point is at x = 0.
left.scatter([0], [0], color="#1E8449", s=200, marker="*", zorder=5)
left.set_title("Bowl: one bottom, always findable")
left.set_xlabel("decision variable")
left.set_ylabel("cost")

# --- Right: the wobbly landscape ---
right = axes[1]
right.plot(x, wobbly, color="#C0392B", linewidth=2.5)
right.set_title("Wobbly: many dips, easy to get stuck")
right.set_xlabel("decision variable")
right.set_ylabel("cost")

plt.show()

## Step 7 — count the traps in the wobbly landscape

Let us actually count the dips. A point is a "local dip" if it is lower than the
point just to its left and just to its right. We walk along the wobbly curve with a
`for` loop and collect them. The true best (the *global* lowest point) is simply the
lowest of all the dips.

In [ ]:
# Collect the position of every point that is lower than both of its neighbours.
local_dip_indexes = []

for i in range(1, len(x) - 1):
    lower_than_left = wobbly[i] < wobbly[i - 1]
    lower_than_right = wobbly[i] < wobbly[i + 1]
    if lower_than_left and lower_than_right:
        local_dip_indexes.append(i)

print("Number of local dips found:", len(local_dip_indexes))

# The global lowest point is the lowest of all the dips.
dip_heights = []
for i in local_dip_indexes:
    dip_heights.append(wobbly[i])

lowest_dip_position = local_dip_indexes[np.argmin(dip_heights)]

print("Best (global) lowest point is at x =", round(x[lowest_dip_position], 2),
      "with cost", round(wobbly[lowest_dip_position], 3))
print()
print("A simple 'roll downhill' search could stop at ANY of these dips,")
print("so on a wobbly problem you cannot be sure you found the best one.")

## What you just did

You drew the two pictures that run through the whole of optimisation:

- a **convex set**, a shape with no dents, where the line between any two points
  stays inside, and
- a **convex function**, a bowl, where every chord lies above the curve.

And you saw the payoff. A convex (bowl-shaped) problem has a single bottom, so any
answer you find is the best one. A wobbly problem can trap you in a dip that only
looks best locally, and you would never know.

The next two notebooks deliver the warm surprise: a whole family of models you have
already met are secretly bowls, so they are all reliably solvable. Next up,
`02_familiar_problems_in_cvxpy.ipynb`.